# 99 Crazyflie Multi-Evaluation

Evaluates all simulation runs found in `simulated_data/rotorpy/` and exports results to `evaluation_data/rotorpy/`.

In [ ]:
from pathlib import Path

from core import SimulationResult
from evaluation import Evaluator
from evaluation.aligners import NearestTimeIndexAligner
from evaluation.metrics import PositionRMSE, NEES
from evaluation.metrics.attitude_rmse import AttitudeRMSE
from evaluation.metrics.velocity_rmse import VelocityRMSE
from evaluation.runners import RustRunnerCrazyflie

SIM_DATA_PATH = Path("../../simulated_data/rotorpy/")
EVAL_DATA_PATH = Path("../../evaluation_data/rotorpy/")
EVAL_DATA_PATH.mkdir(parents=True, exist_ok=True)

sim_pkl_files = sorted(SIM_DATA_PATH.glob("*/*.pkl"))
print(f"Found {len(sim_pkl_files)} simulation(s):")
for p in sim_pkl_files:
    print(f"  {p.stem}")

In [ ]:
VARIANTS = {
    "no_tof": dict(do_correction_tof=False, adaptive_uwb=None, adaptive_tof=None),
    "tof": dict(do_correction_tof=True, adaptive_uwb=None, adaptive_tof=None),
    "tof_sw": dict(do_correction_tof=True, adaptive_uwb="sliding_window", adaptive_tof="sliding_window"),
    "tof_ema": dict(do_correction_tof=True, adaptive_uwb="ema", adaptive_tof="ema"),
}

aligner = NearestTimeIndexAligner()
metrics = [PositionRMSE(), VelocityRMSE(), AttitudeRMSE(), NEES()]

In [ ]:
for variant_name, runner_kwargs in VARIANTS.items():
    variant_dir = EVAL_DATA_PATH / variant_name
    variant_dir.mkdir(parents=True, exist_ok=True)

    runner = RustRunnerCrazyflie(use_noisy=True, do_correction_uwb=True, uwb_stddev=0.1, **runner_kwargs)
    evaluator = Evaluator(runner, metrics, aligner)

    print(f"\n{'#' * 60}")
    print(f"Variant: {variant_name}")
    print(f"{'#' * 60}")

    for pkl_path in sim_pkl_files:
        name = pkl_path.stem
        out_path = variant_dir / f"{name}.pkl"

        print(f"\n{'=' * 60}")
        print(f"  {name}")
        print(f"{'=' * 60}")

        simulation_result = SimulationResult.load(str(pkl_path))
        eval_result = evaluator.evaluate(simulation_result)
        eval_result.save(str(out_path))
        eval_result.summary()
        print(f"→ saved: {out_path}")

print(f"\nDone. {len(VARIANTS)} variant(s) × {len(sim_pkl_files)} scenario(s) exported to {EVAL_DATA_PATH}")

## Visualization

In [ ]:
import plotly.graph_objects as go
import plotly.colors
from evaluation.core import EvaluationResult

PLOT_VARIANT = "tof_ema"
plot_files = sorted((EVAL_DATA_PATH / PLOT_VARIANT).glob("*.pkl"))
plot_results = [EvaluationResult.load(p) for p in plot_files]
scenario_names = [p.stem for p in plot_files]

colors = plotly.colors.qualitative.Plotly
fig = go.Figure()

for i, (result, name) in enumerate(zip(plot_results, scenario_names)):
    color = colors[i % len(colors)]
    gt_pos = result.ground_truth.position  # (N, 3) ENU
    est_pos = result.estimate.position  # (N, 3) ENU

    fig.add_trace(go.Scatter3d(
        x=gt_pos[:, 0], y=gt_pos[:, 1], z=gt_pos[:, 2],
        mode="lines", name=f"{name} (GT)",
        line=dict(width=3, color=color),
        legendgroup=name,
    ))
    fig.add_trace(go.Scatter3d(
        x=est_pos[:, 0], y=est_pos[:, 1], z=est_pos[:, 2],
        mode="lines", name=f"{name} (estimate)",
        line=dict(width=2, color=color, dash="dash"),
        legendgroup=name,
    ))

fig.update_layout(
    title=f"Ground truth vs. estimate — variant: {PLOT_VARIANT}",
    scene=dict(
        xaxis_title="East [m]",
        yaxis_title="North [m]",
        zaxis_title="Up [m]",
        aspectmode="data",
    ),
    margin=dict(l=0, r=0, b=0, t=50),
)
fig.show()